In [ ]:
#Link: https://docs.dgl.ai/en/0.6.x/tutorials/models/1_gnn/1_gcn.html 

In [1]:
import dgl
import dgl.function as fn
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from dgl import DGLGraph

Using backend: pytorch


In [2]:
gcn_msg = fn.copy_u(u='h', out='m')
gcn_reduce = fn.sum(msg='m', out='h')

In [4]:
class GCNLayer(nn.Module):
    def __init__(self, in_feats, out_feats):
        super(GCNLayer, self).__init__()
        self.linear = nn.Linear(in_feats, out_feats)

    def forward(self, g, feature):
        # Creating a local scope so that all the stored ndata and edata
        # (such as the `'h'` ndata below) are automatically popped out
        # when the scope exits.
        with g.local_scope():
            g.ndata['h'] = feature
            g.update_all(gcn_msg, gcn_reduce)
            h = g.ndata['h']
            return self.linear(h)

In [5]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.layer1 = GCNLayer(1433, 16)
        self.layer2 = GCNLayer(16, 7)

    def forward(self, g, features):
        x = F.relu(self.layer1(g, features))
        x = self.layer2(g, x)
        return x
net = Net()
print(net)

Net(
  (layer1): GCNLayer(
    (linear): Linear(in_features=1433, out_features=16, bias=True)
  )
  (layer2): GCNLayer(
    (linear): Linear(in_features=16, out_features=7, bias=True)
  )
)


In [9]:
# We load the cora dataset using DGL’s built-in data module.
from dgl.data import CoraGraphDataset
def load_cora_data():
    dataset = CoraGraphDataset()
    g = dataset[0]
    features = g.ndata['feat']
    labels = g.ndata['label']
    train_mask = g.ndata['train_mask']
    test_mask = g.ndata['test_mask']
    return g, features, labels, train_mask, test_mask

In [7]:
# When a model is trained, we can use the following method to evaluate the performance of the model on the test dataset: 
def evaluate(model, g, features, labels, mask):
    model.eval()
    with th.no_grad():
        logits = model(g, features)
        logits = logits[mask]
        labels = labels[mask]
        _, indices = th.max(logits, dim=1)
        correct = th.sum(indices == labels)
        return correct.item() * 1.0 / len(labels)

In [8]:
import time
import numpy as np
g, features, labels, train_mask, test_mask = load_cora_data()
# Add edges between each node and itself to preserve old node representations
g.add_edges(g.nodes(), g.nodes())
optimizer = th.optim.Adam(net.parameters(), lr=1e-2)
dur = []
for epoch in range(50):
    if epoch >=3:
        t0 = time.time()

    net.train()
    logits = net(g, features)
    logp = F.log_softmax(logits, 1)
    loss = F.nll_loss(logp[train_mask], labels[train_mask])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch >=3:
        dur.append(time.time() - t0)

    acc = evaluate(net, g, features, labels, test_mask)
    print("Epoch {:05d} | Loss {:.4f} | Test Acc {:.4f} | Time(s) {:.4f}".format(
            epoch, loss.item(), acc, np.mean(dur)))

Extracting file to C:\Users\augus\.dgl\cora_v2


D:\ANACONDA\lib\site-packages\scipy\sparse\lil.py:512: FutureWarning: future versions will not create a writeable array from broadcast_array. Set the writable flag explicitly to avoid this warning.
  if not i.flags.writeable or i.dtype not in (np.int32, np.int64):
D:\ANACONDA\lib\site-packages\scipy\sparse\lil.py:514: FutureWarning: future versions will not create a writeable array from broadcast_array. Set the writable flag explicitly to avoid this warning.
  if not j.flags.writeable or j.dtype not in (np.int32, np.int64):


Finished data loading and preprocessing.
  NumNodes: 2708
  NumEdges: 10556
  NumFeats: 1433
  NumClasses: 7
  NumTrainingSamples: 140
  NumValidationSamples: 500
  NumTestSamples: 1000
Done saving data into cached files.


D:\ANACONDA\lib\site-packages\numpy\core\fromnumeric.py:3420: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
D:\ANACONDA\lib\site-packages\numpy\core\_methods.py:188: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Epoch 00000 | Loss 1.9621 | Test Acc 0.3170 | Time(s) nan
Epoch 00001 | Loss 1.8085 | Test Acc 0.4000 | Time(s) nan
Epoch 00002 | Loss 1.6651 | Test Acc 0.4720 | Time(s) nan
Epoch 00003 | Loss 1.5511 | Test Acc 0.5390 | Time(s) 0.0150
Epoch 00004 | Loss 1.4558 | Test Acc 0.5880 | Time(s) 0.0150
Epoch 00005 | Loss 1.3714 | Test Acc 0.6310 | Time(s) 0.0160
Epoch 00006 | Loss 1.2908 | Test Acc 0.6530 | Time(s) 0.0157
Epoch 00007 | Loss 1.2100 | Test Acc 0.6620 | Time(s) 0.0158
Epoch 00008 | Loss 1.1297 | Test Acc 0.6640 | Time(s) 0.0158
Epoch 00009 | Loss 1.0500 | Test Acc 0.6780 | Time(s) 0.0159
Epoch 00010 | Loss 0.9693 | Test Acc 0.6860 | Time(s) 0.0157
Epoch 00011 | Loss 0.8911 | Test Acc 0.6960 | Time(s) 0.0160
Epoch 00012 | Loss 0.8188 | Test Acc 0.7020 | Time(s) 0.0163
Epoch 00013 | Loss 0.7527 | Test Acc 0.7010 | Time(s) 0.0165
Epoch 00014 | Loss 0.6921 | Test Acc 0.7090 | Time(s) 0.0165
Epoch 00015 | Loss 0.6359 | Test Acc 0.7160 | Time(s) 0.0165
Epoch 00016 | Loss 0.5836 | Test 